# 01 · Silver + features (past only)
Attach **lh_fraud** as default lakehouse. Run with `SAMPLE = True` first (300 customers, cheap), then once with `SAMPLE = False`.

Output tables: `silver_transactions`, `gold_features`.
Rule: every feature uses only rows **before** the transaction (`rowsBetween(unboundedPreceding, -1)`, `rangeBetween(-secs, -1)`).

In [51]:
from pyspark.sql import functions as F, Window as W

SAMPLE = False   # True = 300 customers to test the notebook cheaply. False = full run (do it once).
spark.conf.set("spark.sql.shuffle.partitions", "32" if SAMPLE else "200")

StatementMeta(, 7cbc7b6a-2f8e-4d51-9038-3e4c7b116af1, 3, Finished, Available, Finished, False)

## Silver: one clean, joined transaction table (train + test)

In [52]:
tx = spark.table("v_transactions_all")  # keep 'split' column
if SAMPLE:
    keep = spark.table("customers").select("customer_id").orderBy(F.rand(42)).limit(300)
    tx = tx.join(keep, "customer_id")

# Valid = last 2 months of TRAIN only (computed on the full train table, not the sample)
train_max = spark.table("transactions_train").agg(F.max(F.to_date("timestamp")).alias("train_max")).first()["train_max"]
cutoff_date = F.add_months(F.trunc(F.lit(train_max), "month"), -1)   # e.g. 2026-07-31 -> 2026-06-01
print("train max:", train_max)

labels = spark.table("fraud_labels_train")
customers = spark.table("customers").drop("gender")   # gender deliberately not used (fairness)
cards = spark.table("cards").select("card_id", "card_type", "is_international", "daily_limit_tnd",
                                    "atm_daily_limit_tnd", "contactless_enabled")
merchants = spark.table("merchants").select("merchant_id", "merchant_type")

silver = (tx.dropDuplicates(["transaction_id"])
            .join(labels, "transaction_id", "left")
            .join(customers, "customer_id", "left")
            .join(cards, "card_id", "left")
            .join(merchants, "merchant_id", "left")
            .withColumn("amount_tnd", F.col("amount_tnd").cast("double"))
            .withColumn("latitude", F.col("latitude").cast("double"))
            .withColumn("longitude", F.col("longitude").cast("double"))
            .withColumn("ts", F.col("timestamp").cast("long"))
            # New period logic based on 'split' and cutoff
            .withColumn(
                "period",
                F.when(F.col("split") == "test", "test")
                 .when(
                     (F.col("split") == "train") &
                     (F.to_date("timestamp") >= cutoff_date),
                     "valid"
                 )
                 .otherwise("train")
            )
            # Defensive: remove all label columns on test rows
            .withColumn("is_fraud", F.when(F.col("split") == "train", F.col("is_fraud")))
            .withColumn("fraud_scenario", F.when(F.col("split") == "train", F.col("fraud_scenario")))
            .withColumn("fraud_reported_date", F.when(F.col("split") == "train", F.col("fraud_reported_date")))
        )

silver.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("silver_transactions")
s = spark.table("silver_transactions")
display(s.groupBy("period").agg(F.count("*").alias("rows"), F.sum("is_fraud").alias("frauds"),
                                F.round(F.avg("is_fraud") * 100, 3).alias("fraud_pct")).orderBy("period"))

StatementMeta(, 7cbc7b6a-2f8e-4d51-9038-3e4c7b116af1, 4, Finished, Available, Finished, False)

train max: 2026-07-31


SynapseWidget(Synapse.DataFrame, 83498e7b-0150-4772-97e1-5eaed58c7ac6)

In [43]:
# Sanity check: period split in silver_transactions
from pyspark.sql import functions as F

spark.table("silver_transactions").groupBy("period").agg(
    F.count("*").alias("rows"),
    F.sum("is_fraud").alias("frauds"),
    F.round(F.avg("is_fraud") * 100, 3).alias("fraud_pct"),
    F.min("timestamp").alias("first_ts"),
    F.max("timestamp").alias("last_ts")
).orderBy("period").show(truncate=False)

StatementMeta(, 8ff8fabc-fb9c-4464-8326-4b1f1ae2b9bd, 17, Finished, Available, Finished, False)

+------+-------+------+---------+-------------------+-------------------+
|period|rows   |frauds|fraud_pct|first_ts           |last_ts            |
+------+-------+------+---------+-------------------+-------------------+
|test  |187628 |NULL  |NULL     |2026-08-01 00:00:02|2026-09-30 23:58:58|
|train |1542647|8494  |0.551    |2025-01-01 00:01:54|2026-05-31 23:59:33|
|valid |177953 |1013  |0.569    |2026-06-01 00:00:10|2026-07-31 23:58:52|
+------+-------+------+---------+-------------------+-------------------+



## Features: amount vs habits, velocity, novelty, geography, channel, time, profile

In [53]:
s = spark.table("silver_transactions")
HOME = s.groupBy("country").count().orderBy(F.desc("count")).first()["country"]
print("home country value in data:", HOME)

past = W.partitionBy("customer_id").orderBy("ts").rowsBetween(W.unboundedPreceding, -1)
by_cust = W.partitionBy("customer_id").orderBy("ts")


def rng(key, secs):
    return W.partitionBy(key).orderBy("ts").rangeBetween(-secs, -1)


def first_time(*cols):
    w = W.partitionBy("customer_id", *cols).orderBy("ts").rowsBetween(W.unboundedPreceding, -1)
    return (F.count(F.lit(1)).over(w) == 0).cast("int")


def haversine(lat1, lon1, lat2, lon2):
    dlat, dlon = F.radians(lat2 - lat1), F.radians(lon2 - lon1)
    a = F.pow(F.sin(dlat / 2), 2) + F.cos(F.radians(lat1)) * F.cos(F.radians(lat2)) * F.pow(F.sin(dlon / 2), 2)
    return 2 * 6371 * F.asin(F.sqrt(a))

phys = F.col("channel").isin("TPE", "DAB")
foreign = F.coalesce(F.col("country") != HOME, F.lit(False))

f = (
    s.withColumn("key", F.coalesce("card_id", "customer_id"))  # card if any, else customer (transfers)
    .withColumn("is_physical", phys.cast("int"))
    .withColumn("f_declined", (F.col("status") == "REFUSEE").cast("int"))
    .withColumn("f_hour", F.hour("timestamp"))
    .withColumn("f_is_night", (F.col("f_hour") < 6).cast("int"))
    .withColumn("f_dow", F.dayofweek("timestamp"))
    .withColumn(
        "f_payday",
        ((F.dayofmonth("timestamp") >= 25) | (F.dayofmonth("timestamp") <= 3)).cast("int"),
    )
    .withColumn(
        "f_ramadan",
        (
            F.to_date("timestamp").between("2025-03-01", "2025-03-30")
            | F.to_date("timestamp").between("2026-02-18", "2026-03-19")
        ).cast("int"),
    )
    # --- customer history BEFORE this transaction
    .withColumn("f_n_before", F.count(F.lit(1)).over(past))
    .withColumn("f_avg_amt_before", F.avg("amount_tnd").over(past))
    .withColumn("f_std_amt_before", F.stddev("amount_tnd").over(past))
    .withColumn("f_max_amt_before", F.max("amount_tnd").over(past))
    .withColumn("f_night_share_before", F.avg("f_is_night").over(past))
    .withColumn("f_secs_since_prev", F.col("ts") - F.lag("ts").over(by_cust))
    # --- amount
    .withColumn("f_amount", F.col("amount_tnd"))
    .withColumn("f_amount_ratio", F.col("amount_tnd") / F.col("f_avg_amt_before"))
    .withColumn(
        "f_amount_z",
        (F.col("amount_tnd") - F.col("f_avg_amt_before")) / F.col("f_std_amt_before"),
    )
    .withColumn("f_amount_over_max", F.col("amount_tnd") / F.col("f_max_amt_before"))
    .withColumn(
        "f_amount_over_income",
        F.col("amount_tnd") / F.col("monthly_income_tnd").cast("double"),
    )
    .withColumn(
        "f_amount_over_limit",
        F.col("amount_tnd")
        / F.when(F.col("channel") == "DAB", F.col("atm_daily_limit_tnd"))
        .otherwise(F.col("daily_limit_tnd"))
        .cast("double"),
    )
    # --- velocity (past only)
    .withColumn("f_key_tx_10m", F.count(F.lit(1)).over(rng("key", 600)))
    .withColumn("f_key_tx_1h", F.count(F.lit(1)).over(rng("key", 3600)))
    .withColumn("f_key_declines_1h", F.sum("f_declined").over(rng("key", 3600)))
    .withColumn(
        "f_key_small_tx_1h",
        F.sum((F.col("amount_tnd") < 20).cast("int")).over(rng("key", 3600)),
    )
    .withColumn("f_cust_tx_24h", F.count(F.lit(1)).over(rng("customer_id", 86400)))
    .withColumn(
        "f_cust_amt_24h", F.sum("amount_tnd").over(rng("customer_id", 86400))
    )
    # --- novelty
    .withColumn(
        "f_new_merchant",
        F.when(F.col("merchant_id").isNull(), 0).otherwise(first_time("merchant_id")),
    )
    .withColumn(
        "f_new_device",
        F.when(F.col("device_id").isNull(), 0).otherwise(first_time("device_id")),
    )
    .withColumn(
        "f_new_beneficiary",
        F.when(F.col("beneficiary_id").isNull(), 0).otherwise(first_time("beneficiary_id")),
    )
    .withColumn(
        "f_new_ip_country",
        F.when(F.col("ip_country").isNull(), 0).otherwise(first_time("ip_country")),
    )
    .withColumn(
        "f_new_city_phys",
        F.when(
            phys & F.col("city").isNotNull(), first_time("is_physical", "city")
        ).otherwise(0),
    )
    .withColumn(
        "f_new_country_phys",
        F.when(phys, first_time("is_physical", "country")).otherwise(0),
    )
    .withColumn("f_is_foreign", foreign.cast("int"))
    .withColumn(
        "f_ip_foreign",
        F.coalesce(F.col("ip_country") != HOME, F.lit(False)).cast("int"),
    )
    .withColumn(
        "f_same_country_7d",
        F.count(F.lit(1)).over(
            W.partitionBy("customer_id", "country")
            .orderBy("ts")
            .rangeBetween(-7 * 86400, -1)
        ),
    )
    # --- geography: distance / speed since previous located transaction
    .withColumn("prev_lat", F.last("latitude", ignorenulls=True).over(past))
    .withColumn("prev_lon", F.last("longitude", ignorenulls=True).over(past))
    .withColumn(
        "prev_geo_ts",
        F.last(
            F.when(F.col("latitude").isNotNull(), F.col("ts")), ignorenulls=True
        ).over(past),
    )
    .withColumn(
        "f_dist_prev_km",
        haversine(F.col("prev_lat"), F.col("prev_lon"), F.col("latitude"), F.col("longitude")),
    )
    .withColumn(
        "f_speed_kmh",
        F.col("f_dist_prev_km")
        / F.greatest((F.col("ts") - F.col("prev_geo_ts")) / 3600.0, F.lit(0.05)),
    )
    # --- channel / security
    .withColumn("f_magstripe", (F.col("entry_mode") == "MAGSTRIPE").cast("int"))
    .withColumn(
        "f_contactless", (F.col("entry_mode") == "SANS_CONTACT").cast("int")
    )
    .withColumn(
        "f_ecom_no3ds", (F.col("entry_mode") == "ECOM_SANS_3DS").cast("int")
    )
    .withColumn("f_chip_abroad", ((F.col("entry_mode") == "CHIP") & foreign).cast("int"))
    .withColumn(
        "f_contactless_near_limit",
        (
            (F.col("entry_mode") == "SANS_CONTACT")
            & F.col("amount_tnd").between(150, 200)
        ).cast("int"),
    )
    .withColumn(
        "f_card_not_intl_abroad",
        ((F.col("is_international") == False) & foreign).cast("int"),
    )
    .withColumn(
        "f_unusual_night",
        (
            (F.col("f_is_night") == 1)
            & (F.col("f_night_share_before") < 0.05)
            & (F.col("f_n_before") >= 10)
        ).cast("int"),
    )
    # --- profile
    .withColumn("f_age", F.col("age").cast("double"))
    .withColumn(
        "f_income", F.col("monthly_income_tnd").cast("double"),
    )
    .withColumn(
        "f_balance", F.col("avg_monthly_balance_tnd").cast("double"),
    )
    .withColumn(
        "f_tenure_days",
        F.datediff(F.to_date("timestamp"), F.col("customer_since")),
    )
    .withColumn("f_intl_card", F.col("is_international").cast("int"))
    .withColumn("f_digital_user", F.col("digital_banking_user").cast("int"))
    # --- extra coverage
    .withColumn(
        "f_new_category",
        F.when(F.col("merchant_category").isNull(), 0).otherwise(
            first_time("merchant_category")
        ),
    )
    .withColumn(
        "f_key_contactless_24h",
        F.sum((F.col("entry_mode") == "SANS_CONTACT").cast("int")).over(
            rng("key", 86400)
        ),
    )
    .withColumn(
        "f_key_new_merchants_24h",
        F.sum("f_new_merchant").over(rng("key", 86400)),
    )
    .withColumn(
        "f_round_amount",
        (
            (F.col("amount_tnd") >= 100)
            & ((F.col("amount_tnd") % 100) == 0)
        ).cast("int"),
    )
    .withColumn(
        "_pair1",
        F.when(
            F.col("beneficiary_id").isNotNull(),
            (
                F.row_number()
                .over(
                    W.partitionBy("beneficiary_id", "customer_id")
                    .orderBy("ts")
                )
                == 1
            ).cast("int"),
        ),
    )
    .withColumn(
        "f_benef_n_customers_before",
        F.when(
            F.col("beneficiary_id").isNotNull(),
            F.sum("_pair1")
            .over(
                W.partitionBy("beneficiary_id")
                .orderBy("ts")
                .rowsBetween(W.unboundedPreceding, -1)
            ),
        ),
    )
    .drop("prev_lat", "prev_lon", "prev_geo_ts", "_pair1")
)

f = f.cache()
print("rows:", f.count())

StatementMeta(, 7cbc7b6a-2f8e-4d51-9038-3e4c7b116af1, 5, Finished, Available, Finished, False)

home country value in data: TN
rows: 1908228


## Digital-banking signals in the 24 h before MOBILE / WEB / transfer operations

In [54]:
ev = (spark.table("digital_events")
        .select(F.col("customer_id").alias("ev_cust"), F.col("timestamp").cast("long").alias("ets"),
                "event_type", F.col("status").alias("ev_status"), F.col("channel").alias("ev_channel"),
                F.col("beneficiary_id").alias("ev_benef")))

# Transactions where digital context is most relevant
t = (f.filter(F.col("channel").isin("MOBILE", "WEB") | (F.col("transaction_type") == "VIREMENT"))
      .select("transaction_id", "customer_id", "ts", "beneficiary_id"))


def n(cond):
    return F.sum(cond.cast("int"))

# Aggregate digital events in the 24h BEFORE each transaction
# Also keep timing for key events so we can build sequence-pattern flags later
from pyspark.sql import functions as F

dig = (t.join(
            ev,
            (F.col("customer_id") == F.col("ev_cust"))
            & (F.col("ets") < F.col("ts"))
            & (F.col("ets") >= F.col("ts") - 86400),
        )
        .groupBy("transaction_id")
        .agg(
            n((F.col("event_type") == "CONNEXION") & (F.col("ev_status") == "ECHEC")).alias("f_login_fail_24h"),
            n(F.col("event_type") == "AJOUT_APPAREIL").alias("f_device_added_24h"),
            n(F.col("event_type") == "AJOUT_BENEFICIAIRE").alias("f_benef_added_24h"),
            n(F.col("event_type") == "REINITIALISATION_MOT_DE_PASSE").alias("f_pwd_reset_24h"),
            n(F.col("event_type") == "MODIFICATION_NUMERO_TELEPHONE").alias("f_phone_change_24h"),
            n(F.col("ev_channel").isin("AGENCE", "CENTRE_APPEL")).alias("f_branch_events_24h"),
            # Timing to enable sequence-style patterns
            F.min(
                F.when(
                    (F.col("event_type") == "AJOUT_BENEFICIAIRE") & (F.col("ev_benef") == F.col("beneficiary_id")),
                    (F.col("ts") - F.col("ets")) / 60.0,
                )
            ).alias("f_mins_since_benef_added"),
            F.min(
                F.when(
                    F.col("event_type") == "REINITIALISATION_MOT_DE_PASSE",
                    (F.col("ts") - F.col("ets")) / 60.0,
                )
            ).alias("f_mins_since_pwd_reset"),
            F.min(
                F.when(
                    F.col("event_type") == "MODIFICATION_NUMERO_TELEPHONE",
                    (F.col("ts") - F.col("ets")) / 60.0,
                )
            ).alias("f_mins_since_phone_change"),
        ))


StatementMeta(, 7cbc7b6a-2f8e-4d51-9038-3e4c7b116af1, 6, Finished, Available, Finished, False)

## Graph signals: merchant / device shared with frauds **already reported** before the transaction
Only `fraud_reported_date < transaction date` is used (the one legitimate use of that column, brief § 7.1).

In [55]:
# Global smoothed fraud rates (train only) per merchant / device / beneficiary
# These act as risk scores capturing how fraud-prone each entity is.
train_s = s.filter((F.col("period") == "train") & F.col("is_fraud").isNotNull())
base_rate = train_s.agg(F.avg("is_fraud").alias("base")).first()["base"]
prior_n = 100.0  # strength of prior (pseudo-counts)
alpha = base_rate * prior_n
beta = (1.0 - base_rate) * prior_n


def risk_score(key_col, feat_name):
    stats = (train_s.filter(F.col(key_col).isNotNull())
             .groupBy(key_col)
             .agg(F.count("*").alias("tx"), F.sum("is_fraud").alias("frauds")))
    return stats.withColumn(
        feat_name,
        (F.col("frauds") + F.lit(alpha)) / (F.col("tx") + F.lit(alpha + beta)),
    ).select(F.col(key_col), F.col(feat_name))

merchant_risk = risk_score("merchant_id", "f_merchant_risk")
device_risk = risk_score("device_id", "f_device_risk")
benef_risk = risk_score("beneficiary_id", "f_beneficiary_risk")

# Graph signals: merchant / device shared with frauds already reported before the transaction
reported = (s.filter(F.col("is_fraud") == 1)
             .select(
                 "merchant_id",
                 "device_id",
                 "customer_id",
                 "card_id",
                 F.col("fraud_reported_date").alias("rdate"),
             ))


def shared(col, name):
    r = (reported.filter(F.col(col).isNotNull())
                 .groupBy(F.col(col).alias("g_key"), "rdate")
                 .agg(F.count(F.lit(1)).alias("n")))
    base = (f.filter(F.col(col).isNotNull())
              .select(
                  "transaction_id",
                  F.col(col).alias("t_key"),
                  F.to_date("timestamp").alias("d"),
              ))
    return (base.join(
                r,
                (F.col("t_key") == F.col("g_key"))
                & (F.col("rdate") < F.col("d"))
                & (F.col("rdate") >= F.date_sub(F.col("d"), 30)),
            )
                .groupBy("transaction_id")
                .agg(F.sum("n").alias(name)))


# Connectivity-style graph features: distinct fraud customers / cards using same merchant / device in last 30 days

def shared_distinct(key_col, target_col, out_name):
    r = (reported.filter(F.col(key_col).isNotNull() & F.col(target_col).isNotNull())
                 .select(
                     F.col(key_col).alias("g_key"),
                     F.col("rdate"),
                     F.col(target_col).alias("target"),
                 )
                 .dropDuplicates())
    base = (f.filter(F.col(key_col).isNotNull())
              .select(
                  "transaction_id",
                  F.col(key_col).alias("t_key"),
                  F.to_date("timestamp").alias("d"),
              ))
    j = base.join(
        r,
        (F.col("t_key") == F.col("g_key"))
        & (F.col("rdate") < F.col("d"))
        & (F.col("rdate") >= F.date_sub(F.col("d"), 30)),
    )
    return j.groupBy("transaction_id").agg(F.countDistinct("target").alias(out_name))


COUNTS = [
    "f_login_fail_24h",
    "f_device_added_24h",
    "f_benef_added_24h",
    "f_pwd_reset_24h",
    "f_phone_change_24h",
    "f_branch_events_24h",
    "f_merchant_frauds_30d",
    "f_device_frauds_30d",
    "f_merchant_fraud_customers_30d",
    "f_device_fraud_customers_30d",
    "f_merchant_fraud_cards_30d",
    "f_device_fraud_cards_30d",
]


gold = (
    f.join(dig, "transaction_id", "left")
     # Graph counts: how many past reported frauds share this merchant / device in last 30 days
     .join(shared("merchant_id", "f_merchant_frauds_30d"), "transaction_id", "left")
     .join(shared("device_id", "f_device_frauds_30d"), "transaction_id", "left")
     # Graph connectivity: distinct fraud customers / cards sharing same entity in last 30 days
     .join(
         shared_distinct("merchant_id", "customer_id", "f_merchant_fraud_customers_30d"),
         "transaction_id",
         "left",
     )
     .join(
         shared_distinct("device_id", "customer_id", "f_device_fraud_customers_30d"),
         "transaction_id",
         "left",
     )
     .join(
         shared_distinct("merchant_id", "card_id", "f_merchant_fraud_cards_30d"),
         "transaction_id",
         "left",
     )
     .join(
         shared_distinct("device_id", "card_id", "f_device_fraud_cards_30d"),
         "transaction_id",
         "left",
     )
     # Global risk scores per entity (smoothed fraud rates over train period)
     .join(merchant_risk, "merchant_id", "left")
     .join(device_risk, "device_id", "left")
     .join(benef_risk, "beneficiary_id", "left")
)

# Sequence-style pattern flags based on digital timing features
# (0/1 indicators; timing columns kept as numeric features as well)
gold = (
    gold.withColumn(
        "f_pattern_pwd_reset_tx_60m",
        (F.col("f_mins_since_pwd_reset") <= 60).cast("double"),
    )
    .withColumn(
        "f_pattern_phone_change_tx_120m",
        (F.col("f_mins_since_phone_change") <= 120).cast("double"),
    )
)

# Fill missing counts with 0 (risk scores/timings left as-is)
from pyspark.sql import functions as F

gold = gold.fillna(0, subset=COUNTS)

CONTEXT = [
    "transaction_id",
    "timestamp",
    "period",
    "customer_id",
    "card_id",
    "merchant_id",
    "beneficiary_id",
    "device_id",
    "transaction_type",
    "channel",
    "entry_mode",
    "merchant_category",
    "amount_tnd",
    "city",
    "country",
    "ip_country",
    "status",
    "decline_reason",
    "segment",
    "kyc_risk_level",
    "card_type",
    "merchant_type",
    "home_city",
    "is_fraud",
    "fraud_scenario",
]

FEATS = [c for c in gold.columns if c.startswith("f_")]

gold = gold.select(*CONTEXT, *[F.col(c).cast("double").alias(c) for c in FEATS])

gold.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("gold_features")

print(len(FEATS), "features:", FEATS)

f.unpersist()


StatementMeta(, 7cbc7b6a-2f8e-4d51-9038-3e4c7b116af1, 7, Finished, Available, Finished, False)

73 features: ['f_declined', 'f_hour', 'f_is_night', 'f_dow', 'f_payday', 'f_ramadan', 'f_n_before', 'f_avg_amt_before', 'f_std_amt_before', 'f_max_amt_before', 'f_night_share_before', 'f_secs_since_prev', 'f_amount', 'f_amount_ratio', 'f_amount_z', 'f_amount_over_max', 'f_amount_over_income', 'f_amount_over_limit', 'f_key_tx_10m', 'f_key_tx_1h', 'f_key_declines_1h', 'f_key_small_tx_1h', 'f_cust_tx_24h', 'f_cust_amt_24h', 'f_new_merchant', 'f_new_device', 'f_new_beneficiary', 'f_new_ip_country', 'f_new_city_phys', 'f_new_country_phys', 'f_is_foreign', 'f_ip_foreign', 'f_same_country_7d', 'f_dist_prev_km', 'f_speed_kmh', 'f_magstripe', 'f_contactless', 'f_ecom_no3ds', 'f_chip_abroad', 'f_contactless_near_limit', 'f_card_not_intl_abroad', 'f_unusual_night', 'f_age', 'f_income', 'f_balance', 'f_tenure_days', 'f_intl_card', 'f_digital_user', 'f_new_category', 'f_key_contactless_24h', 'f_key_new_merchants_24h', 'f_round_amount', 'f_benef_n_customers_before', 'f_login_fail_24h', 'f_device_add

DataFrame[merchant_id: string, card_id: string, customer_id: string, transaction_id: string, split: string, timestamp: timestamp, transaction_type: string, channel: string, entry_mode: string, merchant_category: string, mcc: int, beneficiary_id: string, amount_tnd: double, original_currency: string, city: string, country: string, latitude: double, longitude: double, device_id: string, ip_country: string, session_id: string, status: string, decline_reason: string, is_fraud: smallint, fraud_scenario: string, fraud_reported_date: date, age: smallint, segment: string, occupation: string, home_city: string, governorate: string, customer_since: date, monthly_income_tnd: decimal(14,3), avg_monthly_balance_tnd: decimal(14,3), kyc_risk_level: string, digital_banking_user: boolean, card_type: string, is_international: boolean, daily_limit_tnd: decimal(12,3), atm_daily_limit_tnd: decimal(12,3), contactless_enabled: boolean, merchant_type: string, ts: bigint, period: string, key: string, is_physic

## Quick sanity check: fraud rate when each binary signal fires (train period)

In [47]:
g = spark.table("gold_features").filter(F.col("period") == "train")
flags = ["f_new_country_phys", "f_magstripe", "f_unusual_night", "f_card_not_intl_abroad", "f_contactless_near_limit",
         "f_new_device", "f_new_beneficiary", "f_ip_foreign"]
display(spark.createDataFrame(
    [(c, *g.filter(F.col(c) == 1).agg(F.count(F.lit(1)), F.avg("is_fraud")).first()) for c in flags],
    "signal string, rows long, fraud_rate double").orderBy(F.desc("fraud_rate")))

StatementMeta(, 8ff8fabc-fb9c-4464-8326-4b1f1ae2b9bd, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6c7c75a9-905f-4dd4-8a48-adca69c4902b)

In [56]:
from pyspark.sql import functions as F

print("=== 1. SILVER: rows / frauds / dates per period ===")
spark.table("silver_transactions").groupBy("period").agg(
    F.count("*").alias("rows"), F.sum("is_fraud").alias("frauds"),
    F.round(F.avg("is_fraud") * 100, 3).alias("fraud_pct"),
    F.min("timestamp").alias("first_ts"), F.max("timestamp").alias("last_ts")
).orderBy("period").show(truncate=False)

g = spark.table("gold_features")
print("=== 2. GOLD: rows / frauds per period ===")
g.groupBy("period").agg(F.count("*").alias("rows"), F.sum("is_fraud").alias("frauds")).orderBy("period").show()

print("=== 3. CHECKS ===")
feats = [c for c in g.columns if c.startswith("f_")]
print("SAMPLE =", SAMPLE)
print("feature count:", len(feats))
print("test rows with a label (must be 0):", g.filter((F.col("period") == "test") & F.col("is_fraud").isNotNull()).count())
print("valid rows (must be > 0):", g.filter(F.col("period") == "valid").count())
print("duplicate transaction_ids (must be 0):", g.count() - g.select("transaction_id").distinct().count())
print("leak columns used as features (must be []):",
      [c for c in feats if any(k in c for k in ["is_fraud", "scenario", "reported", "gender"])])

print("=== 4. FRAUD SCENARIOS in valid ===")
g.filter(F.col("period") == "valid").groupBy("fraud_scenario").count().orderBy(F.desc("count")).show(truncate=False)

StatementMeta(, 7cbc7b6a-2f8e-4d51-9038-3e4c7b116af1, 8, Finished, Available, Finished, False)

=== 1. SILVER: rows / frauds / dates per period ===
+------+-------+------+---------+-------------------+-------------------+
|period|rows   |frauds|fraud_pct|first_ts           |last_ts            |
+------+-------+------+---------+-------------------+-------------------+
|test  |187628 |NULL  |NULL     |2026-08-01 00:00:02|2026-09-30 23:58:58|
|train |1542647|8494  |0.551    |2025-01-01 00:01:54|2026-05-31 23:59:33|
|valid |177953 |1013  |0.569    |2026-06-01 00:00:10|2026-07-31 23:58:52|
+------+-------+------+---------+-------------------+-------------------+

=== 2. GOLD: rows / frauds per period ===
+------+-------+------+
|period|   rows|frauds|
+------+-------+------+
|  test| 187628|  NULL|
| train|1542647|  8494|
| valid| 177953|  1013|
+------+-------+------+

=== 3. CHECKS ===
SAMPLE = False
feature count: 73
test rows with a label (must be 0): 0
valid rows (must be > 0): 177953
duplicate transaction_ids (must be 0): 0
leak columns used as features (must be []): []
=== 4. F

In [49]:
mssparkutils.session.stop()   # free the capacity

StatementMeta(, 8ff8fabc-fb9c-4464-8326-4b1f1ae2b9bd, 23, Finished, Available, Finished, False)